# August 2026 verification · **07 — Gold: Cancer Waiting Times**

**The claims under test (article section 6):**

* 62-day performance **71.9%** against an **85%** standard, "improved year on year and still thirteen points short";
* 28-day Faster Diagnosis **78.6%** against an **80%** operational target, "a target missed by under two points reads as near-success and still means tens of thousands told late".

The gate (04) already rebuilt 71.9% and 78.6% exactly. Here I test the
claims built **on top of** those numbers:

1. the standards themselves, read from NHS England's own file;
2. the gap to standard, in points **and in people**;
3. year on year, like for like (provisional against provisional) and against the revised (final) year-ago month;
4. breach counts and how late the late patients were;
5. the spread between providers: how much the national figure hides.

All figures are provider basis, compliance recomputed as Within / Total.
The reporting month is **provisional**. Outputs go to `data/gold/07_cancer/`.

In [1]:
import math
import re
import sys
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
import pypdf

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import (BRONZE_DIR, GOLD_DIR, PREV_MONTH, REPORT_MONTH, SILVER_DIR,  # noqa: E402
                     YEAR_AGO_MONTH, bronze_file, mon, mon_year, month_name, month_year,
                     months_from, shift_month)

OUT = GOLD_DIR / "07_cancer"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
CWT = BRONZE_DIR / f"nhse_cwt_{REPORT_MONTH}"
CUR, YA = REPORT_MONTH, YEAR_AGO_MONTH
CUR_L, YA_L = mon_year(CUR), mon_year(YA)                                # 'Aug 2026', 'Aug 2025'
CUR_TAG, YA_TAG = f"{mon(CUR).lower()}_{CUR[:4]}", f"{mon(YA).lower()}_{YA[:4]}"   # 'aug_2026', 'aug_2025'
POOL, YA_POOL = months_from(shift_month(CUR, -2), CUR), months_from(shift_month(YA, -2), YA)
POOL_L = f"{mon(POOL[0])}-{mon(CUR)} {CUR[:4]}"                         # 'Jun-Aug 2026'
YA_POOL_L = f"{mon(YA_POOL[0])}-{mon(YA)} {YA[:4]}"                     # 'Jun-Aug 2025' 
MEASURES = ["62D", "FDS"]
NAMES = {"62D": "62-day combined", "FDS": "28-day Faster Diagnosis"}

## Step 0 — The gate must have passed, including the cancer checks

In [2]:
gate = pd.read_csv(GOLD_DIR / "00_reconciliation" / "reconciliation.csv")
assert gate["pass"].all(), f"reconciliation gate has {(~gate['pass']).sum()} failing checks: run 04 first"
assert (gate.gate == "C").sum() >= 14, "the cancer checks are missing from the gate"
cwt = pd.read_parquet(SILVER_DIR / "cwt" / "cwt_provider_headline.parquet")
eng_row = pd.read_parquet(SILVER_DIR / "cwt" / "cwt_england_row.parquet")
prov_ya = pd.read_parquet(SILVER_DIR / "cwt" / f"cwt_{YA}_provisional.parquet")
print(f"gate: {len(gate)} of {len(gate)} checks passed")

gate: 52 of 52 checks passed


## Step 1 — The standards, from NHS England's own file

The national time series states each standard in its header row. I read
the standard from there rather than typing 85 and 80 in.

In [3]:
nts = openpyxl.load_workbook(bronze_file(f"nhse_cwt_{CUR}/CWT-CRS-National-Time-Series-Oct-2009-*-Provisional.xlsx"),
                             read_only=True, data_only=True)["Monthly Performance"]
nts.reset_dimensions()
trows = list(nts.iter_rows(values_only=True))
TITLES = {"FDS": "Four Week (28 days)", "31D": "One Month (31-day)", "62D": "Two Month (62-day)"}
block = {m: next(i for i, x in enumerate(trows[1]) if x and str(x).startswith(t)) for m, t in TITLES.items()}
STANDARD_TEXT = {m: str(trows[2][c]) for m, c in block.items()}
STANDARD = {m: float(re.search(r"= (\d+)%", STANDARD_TEXT[m]).group(1)) / 100 for m in block}
NTS = {(m, r[c - 1].strftime("%Y-%m")): (r[c], r[c + 1]) for r in trows[4:] for m, c in block.items()
       if hasattr(r[c - 1], "strftime")}
print(STANDARD_TEXT)
assert STANDARD["62D"] == 0.85 and STANDARD["FDS"] == 0.80

{'FDS': 'Operational Standard = 80% from Q1 2026/27', '31D': 'Operational Standard = 96% from Q3 2023/24', '62D': 'Operational Standard = 85% from Q3 2023/24'}


## Step 2 — August 2026 against the standard, in points and in people

A percentage gap hides a count. For each standard I compute:

* **late**: patients outside the standard (`After`);
* **shortfall**: how many more patients would have needed to be on time for the standard to be met, `ceil(standard × Total) − Within`;
* **late but allowed**: patients the standard itself permits to be late, `Total − ceil(standard × Total)`.

Late = shortfall + late-but-allowed. That split is what tests "tens of
thousands told late".

In [4]:
nat = cwt.groupby(["measure", "period"])[["total", "within", "after"]].sum()
nat["pct"] = 100 * nat.within / nat.total

def vs_standard(m: str, p: str) -> dict:
    t, w, a = nat.loc[(m, p), ["total", "within", "after"]]
    need = math.ceil(STANDARD[m] * t)
    return {"measure": m, "period": p, "total": t, "within": w, "late": a, "pct": 100 * w / t,
            "standard_pct": 100 * STANDARD[m], "gap_pp": 100 * STANDARD[m] - 100 * w / t,
            "shortfall_people": max(need - w, 0), "late_but_allowed": t - need}

std = pd.DataFrame([vs_standard(m, p) for m in MEASURES for p in [YA, PREV_MONTH, CUR]])
assert (std.shortfall_people + std.late_but_allowed - std.late).abs().max() < 1e-6
std.to_csv(OUT / "vs_standard.csv", index=False)
std

,measure,period,total,within,late,pct,standard_pct,gap_pp,shortfall_people,late_but_allowed
0,62D,2025-09,"30,483.00","20,733.00","9,750.00",68.01,85.00,16.99,"5,178.00","4,572.00"
1,62D,2026-08,"27,909.00","20,066.00","7,843.00",71.90,85.00,13.10,"3,657.00","4,186.00"
2,62D,2026-09,"30,522.00","21,565.00","8,957.00",70.65,85.00,14.35,"4,379.00","4,578.00"
3,FDS,2025-09,"295,202.00","217,972.00","77,230.00",73.84,80.00,6.16,"18,190.00","59,040.00"
4,FDS,2026-08,"273,664.00","214,991.00","58,673.00",78.56,80.00,1.44,"3,941.00","54,732.00"
5,FDS,2026-09,"300,367.00","228,835.00","71,532.00",76.19,80.00,3.81,"11,459.00","60,073.00"


## Step 3 — Year on year, two ways

The reporting month is provisional. The year-ago month exists in two
vintages: as first published (provisional) and as revised (the 2025-26 Final
file). The like-for-like comparison is **provisional against provisional**.
Comparing this month's provisional figure with the year-ago final mixes
vintages. I show both, plus how much the year-ago month moved when it was
finalised. If the revision is similar every year, this month will probably
move the same way.

In [5]:
# Silver keeps the 'ALL ENGLISH PROVIDERS' row beside the trusts; I sum trusts only.
p_ya = prov_ya[prov_ya.org_name.ne("ALL ENGLISH PROVIDERS")].groupby("measure")[["total", "within", "after"]].sum()
LL, MX, RV = f"vs {YA_L} provisional (like for like)", f"vs {YA_L} final (mixed vintages)", f"{YA_L} revision (provisional -> final)"
yoy = []
for m in MEASURES:
    cur = nat.loc[(m, CUR)]
    fin = nat.loc[(m, YA)]
    prv = p_ya.loc[m]
    # The provisional parse must agree with NHS England's provisional national series.
    assert (int(prv.total), int(prv.within)) == tuple(int(x) for x in NTS[(m, YA)]), \
        f"{m}: {YA_L} provisional file != national time series (provisional)"
    for label, b in [(LL, prv), (MX, fin)]:
        yoy.append({"measure": m, "comparison": label,
                    f"{YA_TAG}_pct": 100 * b.within / b.total, f"{CUR_TAG}_pct": 100 * cur.within / cur.total,
                    "change_pp": 100 * (cur.within / cur.total - b.within / b.total),
                    f"{YA_TAG}_total": b.total, f"{CUR_TAG}_total": cur.total,
                    f"late_{YA[:4]}": b.after, f"late_{CUR[:4]}": cur.after, "late_change": cur.after - b.after})
    yoy.append({"measure": m, "comparison": RV,
                f"{YA_TAG}_pct": 100 * prv.within / prv.total, f"{CUR_TAG}_pct": 100 * fin.within / fin.total,
                "change_pp": 100 * (fin.within / fin.total - prv.within / prv.total),
                f"{YA_TAG}_total": prv.total, f"{CUR_TAG}_total": fin.total,
                f"late_{YA[:4]}": prv.after, f"late_{CUR[:4]}": fin.after, "late_change": fin.after - prv.after})
yoy = pd.DataFrame(yoy)
yoy.to_csv(OUT / "year_on_year.csv", index=False)
yoy

,measure,comparison,sep_2025_pct,sep_2026_pct,change_pp,sep_2025_total,sep_2026_total,late_2025,late_2026,late_change
0,62D,vs Sep 2025 provisional (like for like),67.94,70.65,2.72,"30,014.00","30,522.00","9,623.00","8,957.00",-666.00
1,62D,vs Sep 2025 final (mixed vintages),68.01,70.65,2.64,"30,483.00","30,522.00","9,750.00","8,957.00",-793.00
2,62D,Sep 2025 revision (provisional -> final),67.94,68.01,0.08,"30,014.00","30,483.00","9,623.00","9,750.00",127.00
3,FDS,vs Sep 2025 provisional (like for like),73.87,76.19,2.31,"293,004.00","300,367.00","76,548.00","71,532.00","-5,016.00"
4,FDS,vs Sep 2025 final (mixed vintages),73.84,76.19,2.35,"295,202.00","300,367.00","77,230.00","71,532.00","-5,698.00"
5,FDS,Sep 2025 revision (provisional -> final),73.87,73.84,-0.04,"293,004.00","295,202.00","76,548.00","77,230.00",682.00


### The release's 12-month averages

The release gives 12-month averages for FDS and 62-day (in August: 77.3% and
70.2%, September 2025 to August 2026). It doesn't say whether that is the
mean of monthly percentages or the pooled rate, so I compute both and see
which one it is. This also tests the mixed vintages the release itself uses:
final months before April 2026, provisional from April.

In [6]:
rel = re.sub(r"\s+", " ", " ".join(p.extract_text() for p in pypdf.PdfReader(
    bronze_file(f"nhse_cwt_{CUR}/Cancer-Waiting-Times-Statistical-Release-{month_name(CUR)}-{CUR[:4]}-*.pdf")).pages))
REL_AVG = {
    "FDS": float(re.search(r"Performance time series for FDS .*? averaged ([\d.]+)%", rel).group(1)),
    "62D": float(re.search(r"Performance time series for the 62-day Combined standard .*? averaged ([\d.]+)%", rel).group(1)),
}
months12 = months_from(shift_month(CUR, -11), CUR)
avg = []
for m in MEASURES:
    x = nat.loc[m].loc[months12]
    avg.append({"measure": m, "release_says": REL_AVG[m], "mean_of_monthly_pct": x.pct.mean(),
                "pooled_pct": 100 * x.within.sum() / x.total.sum()})
avg = pd.DataFrame(avg)
# I list every method that reproduces the published figure. If both do, the
# release's method can't be told apart at one decimal place, and I say so
# rather than pick one.
avg["methods_that_match"] = avg.apply(lambda r: ", ".join(
    [n for n, v in (("mean of monthly %", r.mean_of_monthly_pct), ("pooled", r.pooled_pct)) if round(v, 1) == r.release_says])
    or "neither", axis=1)
assert (avg.methods_that_match != "neither").all(), "a 12-month average in the release does not reproduce"
avg.to_csv(OUT / "twelve_month_average.csv", index=False)
avg

,measure,release_says,mean_of_monthly_pct,pooled_pct,methods_that_match
0,62D,70.40,70.42,70.44,"mean of monthly %, pooled"
1,FDS,77.50,77.54,77.55,"mean of monthly %, pooled"


## Step 4 — How late were the late patients?

The bands tell me how late, not just whether late. Silver checked they add
up to Total. Here I split the late patients only.

In [7]:
LATE_BANDS = {
    "FDS": {"29-42 days": "In_29_to_42_days", "43-62 days": "In_43_to_62_days", "over 62 days": "After_62_days"},
    "62D": {"63-76 days": "In_63_to_76_days", "77-90 days": "In_77_to_90_days",
            "91-104 days": "In_91_to_104_days", "over 104 days": "After_104_days"},
}
depth = []
for m, bands in LATE_BANDS.items():
    for p in [YA, CUR]:
        e = eng_row[(eng_row.measure == m) & (eng_row.period == p)].iloc[0]
        late_sum = sum(e[c] for c in bands.values())
        assert abs(late_sum - e.after) < 1e-6, f"{m} {p}: late bands != After"
        for label, c in bands.items():
            depth.append({"measure": m, "period": p, "band": label, "patients": e[c], "share_of_late": e[c] / e.after})
depth = pd.DataFrame(depth)
depth.to_csv(OUT / "breach_depth.csv", index=False)
depth.pivot_table(index=["measure", "band"], columns="period", values=["patients", "share_of_late"], sort=False)

patients           share_of_late        
period                  2025-09   2026-09       2025-09 2026-09
measure band                                                   
FDS     29-42 days    38,164.00 37,589.00          0.49    0.53
        43-62 days    24,039.00 21,085.00          0.31    0.29
        over 62 days  15,027.00 12,858.00          0.19    0.18
62D     63-76 days     3,149.00  2,884.00          0.32    0.32
        77-90 days     2,423.00  2,213.00          0.25    0.25
        91-104 days    1,520.00  1,436.00          0.16    0.16
        over 104 days  2,658.00  2,424.00          0.27    0.27

## Step 5 — Provider spread

One month at one trust can be a handful of patients, especially for 62-day.
So I do three things:

* set a **minimum volume** (at least 20 patients in the period) and report how many trusts that drops;
* repeat on the **latest three months pooled** (June-August in the published run), which is steadier;
* give each trust a **Wilson 95% interval**, and count the trusts whose *upper* bound is below the standard, i.e. clearly below it rather than unlucky in a small month.

Trusts are identified by ODS code. The 2026 periods use a single
provisional vintage. For the pooled year-on-year comparison, the year-ago
three months come from the Final file, so that comparison mixes vintages like
the national one does.

In [8]:
MIN_N = 20

def wilson(w, n, z=1.96):
    p = w / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def trust_table(m: str, months: list[str]) -> pd.DataFrame:
    x = cwt[(cwt.measure == m) & cwt.period.isin(months)]
    g = x.groupby("org_code").agg(org_name=("org_name", "last"), total=("total", "sum"),
                                  within=("within", "sum"), after=("after", "sum"), months=("period", "nunique"))
    g = g[g.total > 0]
    g["pct"] = 100 * g.within / g.total
    ci = g.apply(lambda r: wilson(r.within, r.total), axis=1, result_type="expand")
    g["ci_lo"], g["ci_hi"] = 100 * ci[0], 100 * ci[1]
    return g

spread, tables = [], {}
for m in MEASURES:
    for label, months in [(CUR_L, [CUR]), (POOL_L, POOL), (YA_L, [YA]), (YA_POOL_L, YA_POOL)]:
        g = trust_table(m, months)
        tables[(m, label)] = g
        k = g[g.total >= MIN_N]
        s = 100 * STANDARD[m]
        late_sorted = g.after.sort_values(ascending=False)
        spread.append({
            "measure": m, "period": label, "trusts": len(g), f"trusts_n>={MIN_N}": len(k),
            "national_pct": 100 * g.within.sum() / g.total.sum(),
            "min": k.pct.min(), "p10": k.pct.quantile(.1), "median": k.pct.median(),
            "p90": k.pct.quantile(.9), "max": k.pct.max(), "p90_minus_p10": k.pct.quantile(.9) - k.pct.quantile(.1),
            "meeting_standard": int((k.pct >= s).sum()),
            "clearly_below_standard": int((k.ci_hi < s).sum()),
            "share_of_late_at_top20_trusts": late_sorted.head(20).sum() / g.after.sum(),
        })
spread = pd.DataFrame(spread)
spread.to_csv(OUT / "provider_spread.csv", index=False)
spread

,measure,period,trusts,trusts_n>=20,national_pct,min,p10,median,p90,max,p90_minus_p10,meeting_standard,clearly_below_standard,share_of_late_at_top20_trusts
0,62D,Sep 2026,144,125,70.65,48.08,62.08,72.03,84.09,92.13,22.01,10,98,0.39
1,62D,Jul-Sep 2026,150,135,71.22,51.42,62.35,72.45,84.05,100.00,21.70,13,111,0.38
2,62D,Sep 2025,150,129,68.01,36.54,55.46,70.39,84.03,95.92,28.57,10,105,0.37
3,62D,Jul-Sep 2025,157,135,68.95,37.38,58.91,70.78,84.39,100.00,25.48,13,111,0.37
4,FDS,Sep 2026,140,136,76.19,29.63,68.33,79.41,86.76,100.00,18.42,65,57,0.36
5,FDS,Jul-Sep 2026,140,139,78.02,37.14,71.47,80.66,87.75,100.00,16.28,77,57,0.35
6,FDS,Sep 2025,143,134,73.84,53.38,65.13,76.05,84.09,100.00,18.96,39,82,0.34
7,FDS,Jul-Sep 2025,145,139,75.02,16.07,65.29,76.92,85.03,97.92,19.74,45,87,0.33


In [9]:
# Best and worst five trusts, latest three months pooled, at least MIN_N patients.
for m in MEASURES:
    k = tables[(m, POOL_L)]
    k = k[k.total >= MIN_N].sort_values("pct")
    k.to_csv(OUT / f"trusts_{m}_{POOL_L.lower().replace('-', '_').replace(' ', '_')}.csv")
    print(f"== {NAMES[m]}, {POOL_L} pooled, n >= {MIN_N}: lowest and highest five")
    print(pd.concat([k.head(5), k.tail(5)])[["org_name", "total", "pct", "ci_lo", "ci_hi"]].round(1).to_string(), "\n")

== 62-day combined, Jul-Sep 2026 pooled, n >= 20: lowest and highest five
                                                        org_name    total    pct  ci_lo  ci_hi
org_code                                                                                      
RAJ                     MID AND SOUTH ESSEX NHS FOUNDATION TRUST 1,535.50  51.40  48.90  53.90
RHQ            SHEFFIELD TEACHING HOSPITALS NHS FOUNDATION TRUST 1,156.00  51.50  48.60  54.40
RWA                 HULL UNIVERSITY TEACHING HOSPITALS NHS TRUST 1,130.50  54.00  51.10  56.90
RWP                     WORCESTERSHIRE ACUTE HOSPITALS NHS TRUST 1,076.00  56.30  53.30  59.30
RWE                  UNIVERSITY HOSPITALS OF LEICESTER NHS TRUST 1,512.50  56.30  53.80  58.80
RAN                ROYAL NATIONAL ORTHOPAEDIC HOSPITAL NHS TRUST    66.00  90.20  80.60  95.30
NVM                                            EPSOMEDICAL GROUP    24.50  91.80  74.60  97.70
RQ3       BIRMINGHAM WOMEN'S AND CHILDREN'S NHS FOUNDATION TRUST    29.

In [10]:
# Trust-level year on year: year-ago three months (final) vs latest three (provisional), trusts with n >= MIN_N in both.
tyoy = []
for m in MEASURES:
    a, b = tables[(m, YA_POOL_L)], tables[(m, POOL_L)]
    j = a[a.total >= MIN_N][["pct"]].join(b[b.total >= MIN_N][["pct"]], lsuffix=f"_{YA[:4]}", rsuffix=f"_{CUR[:4]}", how="inner")
    j["change_pp"] = j[f"pct_{CUR[:4]}"] - j[f"pct_{YA[:4]}"]
    tyoy.append({"measure": m, "trusts_in_both": len(j), "improved": int((j.change_pp > 0).sum()),
                 "worsened": int((j.change_pp < 0).sum()), "median_change_pp": j.change_pp.median(),
                 "p10_change_pp": j.change_pp.quantile(.1), "p90_change_pp": j.change_pp.quantile(.9)})
tyoy = pd.DataFrame(tyoy)
tyoy.to_csv(OUT / "trust_year_on_year.csv", index=False)
tyoy

,measure,trusts_in_both,improved,worsened,median_change_pp,p10_change_pp,p90_change_pp
0,62D,131,84,47,1.35,-5.43,9.02
1,FDS,134,91,43,2.59,-6.96,10.35


## Step 6 — Verdict, computed

In [11]:
S = std.set_index(["measure", "period"])
Y = yoy.set_index(["measure", "comparison"])
SP = spread.set_index(["measure", "period"])
for m in MEASURES:
    a = S.loc[(m, CUR)]
    print(f"{NAMES[m]} {CUR_L} (provisional): {a.pct:.2f}% vs {a.standard_pct:.0f}% standard -> {a.gap_pp:.2f} pp short; "
          f"{a.late:,.0f} late, of whom {a.late_but_allowed:,.0f} the standard itself allows; "
          f"{a.shortfall_people:,.0f} more on time would have met it.")
    ll, mx, rv = Y.loc[(m, LL)], Y.loc[(m, MX)], Y.loc[(m, RV)]
    print(f"   year on year: {ll.change_pp:+.2f} pp like for like (from {ll[f'{YA_TAG}_pct']:.1f}%); "
          f"{mx.change_pp:+.2f} pp against final; {YA_L} moved {rv.change_pp:+.2f} pp and "
          f"{rv[f'{CUR_TAG}_total'] - rv[f'{YA_TAG}_total']:+,.0f} patients on finalisation.")
    sp = SP.loc[(m, POOL_L)]
    print(f"   providers ({POOL_L}, n>={MIN_N}): {sp[f'trusts_n>={MIN_N}']:.0f} trusts, p10 {sp.p10:.1f}% to p90 {sp.p90:.1f}% "
          f"(range {sp['min']:.1f}-{sp['max']:.1f}%); {sp.meeting_standard:.0f} meet the standard, "
          f"{sp.clearly_below_standard:.0f} are clearly below it; top 20 trusts hold {sp.share_of_late_at_top20_trusts:.0%} of late patients.")
d62 = S.loc[("62D", CUR)]
print(f"\nThe published August claims' tests, applied to {month_year(CUR)}: '13 points short' -> {d62.gap_pp:.1f}; 'improved year on year' -> "
      f"{'yes' if Y.loc[('62D', LL)].change_pp > 0 else 'NO'}; "
      f"FDS 'missed by under two points' -> {S.loc[('FDS', CUR)].gap_pp:.2f}; "
      f"'tens of thousands told late' -> {S.loc[('FDS', CUR)].late:,.0f} late, but only "
      f"{S.loc[('FDS', CUR)].shortfall_people:,.0f} beyond what the 80% standard permits.")

62-day combined Sep 2026 (provisional): 70.65% vs 85% standard -> 14.35 pp short; 8,957 late, of whom 4,578 the standard itself allows; 4,379 more on time would have met it.
   year on year: +2.72 pp like for like (from 67.9%); +2.64 pp against final; Sep 2025 moved +0.08 pp and +469 patients on finalisation.
   providers (Jul-Sep 2026, n>=20): 135 trusts, p10 62.3% to p90 84.1% (range 51.4-100.0%); 13 meet the standard, 111 are clearly below it; top 20 trusts hold 38% of late patients.
28-day Faster Diagnosis Sep 2026 (provisional): 76.19% vs 80% standard -> 3.81 pp short; 71,532 late, of whom 60,073 the standard itself allows; 11,459 more on time would have met it.
   year on year: +2.31 pp like for like (from 73.9%); +2.35 pp against final; Sep 2025 moved -0.04 pp and +2,198 patients on finalisation.
   providers (Jul-Sep 2026, n>=20): 139 trusts, p10 71.5% to p90 87.8% (range 37.1-100.0%); 77 meet the standard, 57 are clearly below it; top 20 trusts hold 35% of late patients.

The 